# Percentiles, the five-number summary and box plots
Every calculation of the Note, in code.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats

marks = np.array([78, 82, 84, 88, 91, 93, 94, 96, 98, 99])

## 1. The value at a percentile, by hand and with NumPy

In [ ]:
def percentile_np1(x, p):
    """The (n + 1) rule: location p/100 * (n + 1), then move part of the way to the next value."""
    x = np.sort(x)
    loc = p / 100 * (len(x) + 1)
    k, d = int(loc), loc - int(loc)          # whole part and decimal part
    if k < 1:
        return x[0]
    if k >= len(x):
        return x[-1]
    return x[k - 1] + d * (x[k] - x[k - 1])  # positions count from 1, Python from 0

for p in [25, 50, 75, 90]:
    print(p, percentile_np1(marks, p), np.percentile(marks, p, method='weibull'), np.percentile(marks, p))

The last column is NumPy's default (`linear`) rule: on small data it differs a little.

## 2. The percentile of a value

In [ ]:
def percentile_rank(x, v):
    below = (x < v).sum()                 # X: values below v
    equal = (x == v).sum()                # Y: values equal to v
    return (below + 0.5 * equal) / len(x) * 100

for v in [88, 99]:
    print(v, percentile_rank(marks, v), stats.percentileofscore(marks, v, kind='mean'))

## 3. Building a box plot by hand

In [ ]:
data = np.array([6, 213, 241, 260, 281, 290, 314, 321, 350, 1500])
q1, q2, q3 = (percentile_np1(data, p) for p in (25, 50, 75))
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
inside = data[(data >= lo) & (data <= hi)]
print('Q1, Q2, Q3:', q1, q2, q3, ' IQR:', iqr)
print('fences:', lo, hi)
print('whiskers end at', inside.min(), 'and', inside.max())
print('outliers:', data[(data < lo) | (data > hi)])

In [ ]:
px.box(x=data, points='outliers')

## 4. Side-by-side box plots

In [ ]:
df = pd.read_csv('data/titanic_train.csv')
print(df.groupby('Pclass')['Age'].describe()[['min', '25%', '50%', '75%', 'max']])
px.box(df, x='Pclass', y='Age', points='outliers')